In [1]:
!pip -q install -U "mcp>=2,<3" google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [2]:
from mcp.server import MCPServer

mcp = MCPServer("Online Store")

products = {
    "iphone": {
        "price": 999,
        "stock": 10
    },
    "samsung": {
        "price": 799,
        "stock": 5
    },
    "macbook": {
        "price": 1499,
        "stock": 3
    }
}

@mcp.tool()
def get_product_price(product_name: str) -> str:
    """Get the current price of a product."""

    product = products.get(product_name.lower())

    if not product:
        return "Product not found"

    return f"${product['price']}"


@mcp.tool()
def check_stock(product_name: str) -> str:
    """Check the available stock of a product."""

    product = products.get(product_name.lower())

    if not product:
        return "Product not found"

    return f"{product['stock']} items available"


print("MCP Server created successfully!")

MCP Server created successfully!


In [3]:
from mcp import Client

async def show_tools():

    async with Client(mcp) as client:

        result = await client.list_tools()

        for tool in result.tools:
            print("Tool:", tool.name)
            print("Description:", tool.description)
            print()

await show_tools()

Tool: get_product_price
Description: Get the current price of a product.

Tool: check_stock
Description: Check the available stock of a product.



In [6]:
async def test_tool():

    async with Client(mcp) as client:

        result = await client.call_tool(
            "get_product_price",
            {"product_name": "iphone"}
        )

        for item in result.content:
            if item.type == "text":
                print(item.text)

await test_tool()

$999


In [7]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")

gemini = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini initialized!")

Gemini initialized!


In [8]:
response = gemini.models.generate_content(
    model=MODEL,
    contents="What is MCP? Explain in one sentence."
)

print(response.text)

MCP (Model Context Protocol) is an open standard created by Anthropic that allows AI models to securely connect and interact with external data sources and tools.


In [9]:
import json
from google.genai import types
from mcp import Client

async def ask_store(question):

    async with Client(mcp) as client:

        # Get available MCP tools
        tools_result = await client.list_tools()

        available_tools = [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.input_schema
            }
            for tool in tools_result.tools
        ]

        # Ask Gemini to choose a tool
        prompt = f"""
        You are an AI assistant for an online store.

        Available MCP tools:
        {json.dumps(available_tools)}

        Customer question:
        {question}

        Choose the best tool for the question.

        Return a JSON object with:
        - tool: tool name, or null if no tool is needed
        - arguments: object containing tool arguments

        Only use tools from the available list.
        Product names are iphone, samsung, and macbook.
        If no available tool can answer the question,
        return null for tool and an empty arguments object.
        """

        decision = gemini.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json"
            )
        )

        selected = json.loads(decision.text)

        tool_name = selected.get("tool")
        arguments = selected.get("arguments", {})

        if tool_name is None:
            print("No suitable MCP tool found.")
            return

        allowed = {tool.name for tool in tools_result.tools}

        if tool_name not in allowed:
            print("Gemini selected an unknown tool.")
            return

        if not isinstance(arguments, dict):
            print("Invalid tool arguments.")
            return

        print("Selected MCP Tool:", tool_name)
        print("Arguments:", arguments)

        # Execute the selected MCP tool
        result = await client.call_tool(
            tool_name,
            arguments
        )

        tool_output = "\n".join(
            item.text
            for item in result.content
            if item.type == "text"
        )

        if result.is_error:
            print("MCP Tool Error:", tool_output)
            return

        print("MCP Result:", tool_output)

        # Ask Gemini to explain the tool result
        final_response = gemini.models.generate_content(
            model=MODEL,
            contents=f"""
            Customer question: {question}

            Tool result: {tool_output}

            Answer the customer in a short, friendly way.
            Use only the tool result for product information.
            """
        )

        print("\nGemini:", final_response.text)

In [10]:
await ask_store("What is the price of an iPhone?")

Selected MCP Tool: get_product_price
Arguments: {'product_name': 'iphone'}
MCP Result: $999

Gemini: The iPhone is $999! Let me know if you need any more help.


In [17]:
await ask_store("How many MacBooks are available?")

Selected MCP Tool: check_stock
Arguments: {'product_name': 'macbook'}
MCP Result: 3 items available

Gemini: We currently have 3 MacBooks available!


In [23]:
await ask_store("May I  know the price of the latest apple devices?")

Selected MCP Tool: get_product_price
Arguments: {'product_name': 'iphone'}
MCP Result: $999

Gemini: The price is $999! Let me know if you need anything else.
